<a href="https://colab.research.google.com/github/codedbyrajput/PyTorch-Labs/blob/main/Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IN lab 1 we only had 1 neuron without activation function due to which we would only get a straight line however in real life scenarios we not always have a straight line

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import helper_utils

# Combined dataset: bikes for short distances, cars for longer ones
distances = torch.tensor([
    [1.0], [1.5], [2.0], [2.5], [3.0], [3.5], [4.0], [4.5], [5.0], [5.5],
    [6.0], [6.5], [7.0], [7.5], [8.0], [8.5], [9.0], [9.5], [10.0], [10.5],
    [11.0], [11.5], [12.0], [12.5], [13.0], [13.5], [14.0], [14.5], [15.0], [15.5],
    [16.0], [16.5], [17.0], [17.5], [18.0], [18.5], [19.0], [19.5], [20.0]
], dtype=torch.float32)

# Corresponding delivery times in minutes
times = torch.tensor([
    [6.96], [9.67], [12.11], [14.56], [16.77], [21.7], [26.52], [32.47], [37.15], [42.35],
    [46.1], [52.98], [57.76], [61.29], [66.15], [67.63], [69.45], [71.57], [72.8], [73.88],
    [76.34], [76.38], [78.34], [80.07], [81.86], [84.45], [83.98], [86.55], [88.33], [86.83],
    [89.24], [88.11], [88.16], [91.77], [92.27], [92.13], [90.73], [90.39], [92.98]
], dtype=torch.float32)

Architecture upgrade
Now we will do something like this:
model = nn.Sequential(
    nn.Linear(1, 3), # Hidden Layer
    nn.ReLU(),       # The Hinge / Bouncer
    nn.Linear(3, 1)  # Output Layer
)
Here instead of making a final guess right away, we will be using 3 neurons with their own regression equations and own unique straight lines.

Now ReLu sees if any part of the line drops below 0, it makes it 0 ehich gives a sharp hinge or bend in the lines.

The output layer takes those 3 separate bent lines, weights their importance, and adds them all together. By combining multiple bent lines, weights their importance and adds them all together. By combining multiple bent lines, your model can now draw a continuous, custom curve that perfectly hugs your data points.

Normalization

distances_norm = (distances - distances_mean) / distances_std
This is called Z-Score Standardization. Machine learning models are heavily reliant on the math of gradient descent (that "valley" we talked about earlier).

When you feed a neural network raw numbers that are vastly different in scale—like a distance of 2.0 and a time of 92.0—the math can become unstable. The "valley" becomes stretched and warped, making it very hard for the optimizer to find the bottom.

By subtracting the mean and dividing by the standard deviation, you "squish" all your data so it is centered around 0 (usually between -2 and +2). The physical shape of the curve stays exactly the same, but the numbers are now small and perfectly balanced, allowing the optimizer to learn much faster.

In [ ]:
# Calculate the mean and standard deviation for the 'distances' tensor
distances_mean = distances.mean()
distances_std = distances.std()

# Calculate the mean and standard deviation for the 'times' tensor
times_mean = times.mean()
times_std = times.std()

# Apply standardization to the distances.
distances_norm = (distances - distances_mean) / distances_std

# Apply standardization to the times.
times_norm = (times - times_mean) / times_std

Building NON LINEAR MODEL


In [ ]:
torch.manual_seed(27)

model = nn.Sequential(
    nn.Linear(1, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

Training

Define the loss function and the optimizer for training.

In [ ]:
# Define the loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
# Training loop
for epoch in range(3000):
    # Reset the optimizer's gradients
    optimizer.zero_grad()
    # Make predictions (forward pass)
    outputs = model(distances_norm)
    # Calculate the loss
    loss = loss_function(outputs, times_norm)
    # Calculate adjustments (backward pass)
    loss.backward()
    # Update the model's parameters
    optimizer.step()

    # Create a live plot every 50 epochs
    if (epoch + 1) % 50 == 0:
        helper_utils.plot_training_progress(
            epoch=epoch,
            loss=loss,
            model=model,
            distances_norm=distances_norm,
            times_norm=times_norm
        )

print("\nTraining Complete.")
print(f"\nFinal Loss: {loss.item()}")

Make Predictions

Because you trained the model on "squished" normalized data, the model's brain only understands that squished scale.

If you ask the trained model, "How long for a 5.1-mile delivery?", and just pass it the number 5.1, the model will panic. On its normalized scale, 5.1 is completely off the charts!

That is why the prediction block at the end of Lab 2 has extra steps:

Translate the Question: You have to squish your 5.1 input using the exact same math you used during training (new_distance_norm).

Ask the Model: The model gives you a predicted time, but the answer is also squished (e.g., -0.5).

Translate the Answer: You have to reverse the math (predicted_time_norm * times_std + times_mean) to stretch that -0.5 back out into a real-world number like 38.7 minutes.

In [ ]:
distance_to_predict = 5.1
# Use the torch.no_grad() context manager for efficient prediction
with torch.no_grad():
    # Normalize the input distance
    distance_tensor = torch.tensor([[distance_to_predict]], dtype=torch.float32)
    new_distance_norm = (distance_tensor - distances_mean) / distances_std

    # Get the normalized prediction from the model
    predicted_time_norm = model(new_distance_norm)

    # De-normalize the output to get the actual time in minutes
    predicted_time_actual = (predicted_time_norm * times_std) + times_mean

    # --- Decision Making Logic ---
    print(f"Prediction for a {distance_to_predict}-mile delivery: {predicted_time_actual.item():.1f} minutes")

    # First, check if the delivery is possible within the 45-minute timeframe
    if predicted_time_actual.item() > 45:
        print("\nDecision: Do NOT promise the delivery in under 45 minutes.")
    else:
        # If it is possible, then determine the vehicle based on the distance
        if distance_to_predict <= 3:
            print(f"\nDecision: Yes, delivery is possible. Since the distance is {distance_to_predict} miles (<= 3 miles), use a bike.")
        else:
            print(f"\nDecision: Yes, delivery is possible. Since the distance is {distance_to_predict} miles (> 3 miles), use a car.")